In [ ]:
# tmux launch screen
# 1. Locations of the already-downloaded packages
CACHE="/data1/liborui/.conda/pkgs"
NCURSES="$CACHE/ncurses-6.6-hdb14827_1"
TMUX_BIN="$CACHE/tmux-3.8_-hd13ee6d_0/bin/tmux"

# 2. Configure the required shared libraries
export LD_LIBRARY_PATH="$NCURSES/lib:${LD_LIBRARY_PATH:-}"

# 3. Locate available terminfo databases
TERMINFO_PATHS=""

for dir in \
    "$NCURSES/share/terminfo" \
    "$NCURSES/lib/terminfo" \
    /opt/anaconda3/share/terminfo \
    /usr/share/terminfo \
    /lib/terminfo \
    /etc/terminfo \
    "$HOME/.terminfo"
do
    if [ -d "$dir" ]; then
        echo "[FOUND] Terminfo directory: $dir"
        TERMINFO_PATHS="${TERMINFO_PATHS:+$TERMINFO_PATHS:}$dir"
    fi
done

# Also search cached Conda packages
while IFS= read -r dir; do
    case ":$TERMINFO_PATHS:" in
        *":$dir:"*) ;;
        *) TERMINFO_PATHS="${TERMINFO_PATHS:+$TERMINFO_PATHS:}$dir" ;;
    esac
done < <(find "$CACHE" -type d -path '*/share/terminfo' 2>/dev/null)

if [ -z "$TERMINFO_PATHS" ]; then
    echo "[ERROR] No terminfo directory found."
else
    # 4. Configure ncurses
    unset TERMINFO
    export TERMINFO_DIRS="$TERMINFO_PATHS"

    # 5. Check the current terminal type
    echo "Current TERM: ${TERM:-unset}"

    # Use xterm-256color if the existing type is unavailable
    if ! find ${TERMINFO_DIRS//:/ } -type f -name "${TERM:-}" \
        -print -quit 2>/dev/null | grep -q .; then
        export TERM=xterm-256color
        echo "[INFO] Using TERM=$TERM"
    fi

    # 6. Verify tmux
    "$TMUX_BIN" -V

    # 7. Start OpenClaw session
    "$TMUX_BIN" new -s test
fi
tmux attach -t test